In [16]:
import os
from argparse import ArgumentParser

import torch
from torch.utils.data import DataLoader

from neuralut.nn import (
    generate_truth_tables,
    lut_inference,
    module_list_to_verilog_module,
)

from train import configs, model_config, dataset_config, test
from dataset import JetSubstructureDataset
from models import JetSubstructureNeqModel, JetSubstructureLutModel
from neuralut.synthesis import synthesize_and_get_resource_counts

cuda_device = 2
config = configs["jsc-cernbox"]
torch.cuda.set_device(cuda_device)

In [17]:
# Fetch the test set
dataset = {}
dataset["test"] = JetSubstructureDataset(
    "data/processed-pythia82-lhc13-all-pt1-50k-r1_h022_e0175_t220_nonu_truth.z",
    "config/yaml_IP_OP_config.yml",
    split="test",
)
test_loader = DataLoader(
    dataset["test"], batch_size=config["batch_size"], shuffle=False
)

In [18]:
imask = torch.load("./test_demo/imask.pth", map_location="cuda:{}".format(cuda_device))

In [19]:
model_cfg = {}
for k in model_config.keys():
    model_cfg[k] = config[k]
model_cfg["input_length"] = 16
model_cfg["output_length"] = 5
model_cfg["imask"] = torch.load("./test_demo/imask.pth", map_location="cuda:{}".format(cuda_device))
model_cfg['dense_forward'] = False
model_cfg["cuda"] = True

In [20]:
model = JetSubstructureNeqModel(model_cfg)
model.cuda()

Creating input layer
Creating support layer
Creating support layer
Creating support layer
Creating support layer
Creating support layer
Creating output layer


JetSubstructureNeqModel(
  (module_list): ModuleList(
    (0): SparseLinearNeq(
      (input_quant): QuantBrevitasActivation(
        (brevitas_module): QuantHardTanh(
          (input_quant): ActQuantProxyFromInjector(
            (_zero_hw_sentinel): StatelessBuffer()
          )
          (act_quant): ActQuantProxyFromInjector(
            (_zero_hw_sentinel): StatelessBuffer()
            (fused_activation_quant_proxy): FusedActivationQuantProxy(
              (activation_impl): Identity()
              (tensor_quant): RescalingIntQuant(
                (int_quant): IntQuant(
                  (float_to_int_impl): RoundSte()
                  (tensor_clamp_impl): TensorClamp()
                  (delay_wrapper): DelayWrapper(
                    (delay_impl): _NoDelay()
                  )
                )
                (scaling_impl): ParameterScaling(
                  (restrict_clamp_scaling): _RestrictClampValue(
                    (clamp_min_ste): Identity()
               

In [21]:
checkpoint = torch.load("./test_demo/best_accuracy.pth", map_location="cuda:{}".format(cuda_device))
model.load_state_dict(checkpoint["model_dict"])

<All keys matched successfully>

In [22]:
# Test the PyTorch model
print("Running inference on baseline model...")
baseline_accuracy = test(model, test_loader, cuda=True)
print("Baseline accuracy: %f" % (baseline_accuracy))

Running inference on baseline model...


Baseline accuracy: 74.982013


In [23]:
# Generate the truth tables in the LUT module
print("Converting to NEQs to LUTs...")
generate_truth_tables(model, verbose=True)

Converting to NEQs to LUTs...
Calculating truth tables for module_list.0
Truth tables generated for 320 neurons
Calculating truth tables for module_list.1
Truth tables generated for 160 neurons
Calculating truth tables for module_list.2
Truth tables generated for 80 neurons
Calculating truth tables for module_list.3
Truth tables generated for 40 neurons
Calculating truth tables for module_list.4
Truth tables generated for 20 neurons
Calculating truth tables for module_list.5
Truth tables generated for 10 neurons
Calculating truth tables for module_list.6
Truth tables generated for 5 neurons


In [24]:
# Test the LUT-based model
print("Running inference on LUT-based model...")
lut_inference(model)
lut_accuracy = test(model, test_loader, cuda=True)
print("LUT-Based Model accuracy: %f" % (lut_accuracy))
modelSave = {"model_dict": model.state_dict(), "test_accuracy": lut_accuracy}
torch.save(modelSave, "./test_demo/verilog/" + "/lut_based_model.pth")

Running inference on LUT-based model...
LUT-Based Model accuracy: 74.989106


In [25]:
print("Generating verilog in %s..." % ("./test_demo/verilog/"))
module_list_to_verilog_module(
    model.module_list,
    "neuralut",
    "./test_demo/verilog/",
    add_registers=True,
)
print("Top level entity stored at: %s/neuralut.v ..." % ("./test_demo/verilog/"))

Generating verilog in ./test_demo/verilog/...
Top level entity stored at: ./test_demo/verilog//neuralut.v ...


In [26]:
import os

os.environ["OHMYXILINX"] = "/home/ma11418/NeuraLUT_Private/NeuraLUT_Private/oh-my-xilinx"

In [27]:
print("Running out-of-context synthesis")
ret = synthesize_and_get_resource_counts("./test_demo/verilog/", "neuralut", fpga_part='xcvu9p-flgb2104-2-i', clk_period_ns='1.1', post_synthesis=1)
print("Max f: " + str(ret))

Running out-of-context synthesis


/home/ma11418/NeuraLUT_Private/NeuraLUT_Private/oh-my-xilinx/vivadoprojgen.sh:31: no matches found: ../*.vhd
/home/ma11418/NeuraLUT_Private/NeuraLUT_Private/oh-my-xilinx/vivadoprojgen.sh:33: no matches found: ../*.h
/home/ma11418/NeuraLUT_Private/NeuraLUT_Private/oh-my-xilinx/vivadoprojgen.sh:34: no matches found: ../*.xdc
/home/ma11418/NeuraLUT_Private/NeuraLUT_Private/oh-my-xilinx/vivadoprojgen.sh:35: no matches found: ../*.vh
cat: neuralut.xdc: input file is output file


['LUT', '8574']
['FF', '2706']
['DSP', '0']
['BRAM', '0']
['WNS', '0.036']
['']
Max f: {'vivado_proj_folder': './test_demo/verilog//results_neuralut', 'LUT': 8574.0, 'FF': 2706.0, 'DSP': 0.0, 'BRAM': 0.0, 'WNS': 0.036, '': 0, 'fmax_mhz': 939.8496240601503}
